# FAISS: retrieve relevant documents and measure the search tradeoff

A help center has articles, but readers arrive with short questions rather than exact titles. We will turn documents into vectors, build an exact similarity index, inspect returned IDs, evaluate a small labeled query set and compare exact search with approximate indexes on a separate numerical workload.

FAISS searches numerical vectors. It does not create language embeddings, store a full document database or decide which results a user is allowed to see. The default project uses local TF-IDF and SVD representations so it runs offline after installation. These are **lexical latent vectors, not pretrained semantic embeddings**.

The teaching data consists of 20 fictional articles and 6 authored query judgments. A separate seeded matrix supports an approximation demonstration. All code, examples, evaluation and reference notes are in this notebook.

## 1. Make the numerical contract visible

Install `faiss-cpu==1.12.0` with the repository's numpy, scikit-learn, pandas, matplotlib and joblib. The Python import is `faiss`. This notebook uses CPU search and two OpenMP threads; it makes no GPU assumptions.

Files are written under `outputs/faiss/` and replaced on rerun. A valid search input has a known dimension, finite values, the expected dtype and the same preprocessing as the indexed documents. We will verify those properties before trusting neighbors.

In [ ]:
from pathlib import Path
import time, json, platform
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import faiss
import joblib
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import normalize
from IPython.display import display
OUT = Path('outputs/faiss'); OUT.mkdir(parents=True, exist_ok=True)
faiss.omp_set_num_threads(2)
print({'python':platform.python_version(), 'faiss':faiss.__version__, 'numpy':np.__version__})

## 2. Decide which documents belong in the searchable population

One row is a help-center article with an integer ID, title, body, category, tenant, publication status and revision. Before similarity search, select published articles belonging to tenant alpha. Another tenant's nearly identical article is still the wrong result for this population.

The query judgments contain one designated relevant article each. We will report development and test results separately, but these six authored examples are far too few for a production retrieval assessment. The corpus is available to the vectorizer; query labels are not used to fit it.

In [ ]:
# Fictional help-center articles authored for this notebook; no scraped or private content.
base_articles = [
 ('Reset a password', 'Reset a forgotten password using the account recovery email link.', 'account'),
 ('Two factor recovery', 'Recover two factor authentication access with backup codes when your phone is lost.', 'account'),
 ('Change an email address', 'Change your account email address after confirming the new email inbox.', 'account'),
 ('Unlock an account', 'Unlock a locked account after too many failed login attempts.', 'account'),
 ('Download an invoice', 'Download a billing invoice and receipt from the payment history page.', 'billing'),
 ('Duplicate payment', 'Report a duplicate card charge when the same payment appears twice.', 'billing'),
 ('Refund status', 'Track a refund status and check when money returns to your card.', 'billing'),
 ('Update a payment card', 'Update an expired credit card before the next subscription payment.', 'billing'),
 ('Track a shipment', 'Track a shipment with the package tracking number and courier link.', 'delivery'),
 ('Change a delivery address', 'Change the delivery address before a package leaves the warehouse.', 'delivery'),
 ('Damaged package', 'Report a damaged package with photos of the parcel and broken items.', 'delivery'),
 ('Missing delivery', 'Investigate a missing delivery when tracking says delivered but the package is absent.', 'delivery'),
 ('Cancel a subscription', 'Cancel a recurring subscription to stop future renewal charges.', 'subscription'),
 ('Pause a subscription', 'Pause a subscription temporarily and resume the plan next month.', 'subscription'),
 ('Upgrade a plan', 'Upgrade your subscription plan to increase storage and team member limits.', 'subscription'),
 ('Internal refund draft', 'Refund money instantly without review; this fictional draft is not published policy.', 'billing')
]
records = []
for i,(title,text,category) in enumerate(base_articles):
    records.append({'id':1001+i, 'title':title, 'text':text, 'category':category,
                    'tenant':'alpha', 'status':'draft' if i == 15 else 'published', 'revision':1})
# A second fictional tenant reuses four topics. Metadata must distinguish ownership.
for i in [0,4,8,12]:
    row = dict(records[i]); row.update(id=2001+i, tenant='beta', title='Beta: '+row['title'])
    records.append(row)
docs = pd.DataFrame(records)
assert docs.id.is_unique and len(docs) == 20
queries = pd.DataFrame([
    {'query':'forgotten password recovery email', 'relevant_id':1001, 'split':'development'},
    {'query':'duplicate card charge payment twice', 'relevant_id':1006, 'split':'development'},
    {'query':'track shipment courier number', 'relevant_id':1009, 'split':'development'},
    {'query':'phone lost two factor backup codes', 'relevant_id':1002, 'split':'test'},
    {'query':'damaged parcel broken items photos', 'relevant_id':1011, 'split':'test'},
    {'query':'pause subscription resume next month', 'relevant_id':1014, 'split':'test'}
])

catalog = docs[(docs.tenant == 'alpha') & (docs.status == 'published')].reset_index(drop=True)
assert len(catalog) == 15
metadata = catalog.set_index('id')
display(catalog[['id','title','category']])
display(queries)

## 3. Separate text representation from nearest-neighbor search

TF-IDF weights corpus words and word pairs. SVD compresses that sparse representation into 10 dense dimensions. Normalization puts nonzero vectors on the unit sphere. FAISS will rank the resulting numbers; it does not know the text that produced them.

Fitting the representation on the searchable corpus is appropriate for this fixed-corpus retrieval demonstration. For a changing corpus, version the vocabulary and SVD transform and rebuild incompatible indexes. Query words unseen in the corpus cannot contribute lexical evidence.

Predict what will happen for a query containing only unknown terms. Reject zero vectors instead of reporting arbitrary zero-score neighbors as useful results.

In [ ]:
vectorizer = TfidfVectorizer(ngram_range=(1,2))
text_matrix = vectorizer.fit_transform(catalog.title + ' ' + catalog.text)
svd = TruncatedSVD(n_components=10, random_state=42)
vectors = np.ascontiguousarray(normalize(svd.fit_transform(text_matrix)), dtype=np.float32)
assert vectors.shape == (15,10) and np.isfinite(vectors).all()
assert np.allclose(np.linalg.norm(vectors,axis=1),1,atol=1e-5)

def encode_queries(texts):
    sparse = vectorizer.transform(texts)
    dense = svd.transform(sparse)
    if (np.linalg.norm(dense,axis=1) < 1e-10).any():
        raise ValueError('At least one query has no usable corpus vocabulary; ask for a more specific query.')
    return np.ascontiguousarray(normalize(dense),dtype=np.float32)

try:
    encode_queries(['zzzzunknownword'])
except ValueError as exc:
    print('Expected empty-evidence check:',exc)
print('Sparse vocabulary size:',text_matrix.shape[1], 'Dense dimension:',vectors.shape[1])

## 4. Choose a metric with an interpretable score

`IndexFlatIP` performs exact inner-product search. For unit-normalized document and query vectors, inner product equals cosine similarity. Larger scores are better. `IndexFlatL2` returns **squared** Euclidean distances, for which smaller is better.

For unit vectors, squared L2 distance equals `2 − 2 × inner_product`. Verify that identity on a tiny example before using it in the project. Without normalization, inner product can favor large vector norms; it is not automatically cosine similarity.

In [ ]:
toy = np.ascontiguousarray([[1,0],[0,1],[-1,0]], dtype=np.float32)
query_vector = np.ascontiguousarray([[1,0]], dtype=np.float32)
ip = faiss.IndexFlatIP(2); ip.add(toy)
l2 = faiss.IndexFlatL2(2); l2.add(toy)
similarity, ip_ids = ip.search(query_vector,3)
distance, l2_ids = l2.search(query_vector,3)
assert np.array_equal(ip_ids,l2_ids)
assert np.allclose(distance,2-2*similarity)
display(pd.DataFrame({'row':ip_ids[0], 'cosine':similarity[0], 'squared_L2':distance[0]}))

## 5. Preserve document IDs when rows move

A plain flat index returns row positions. `IndexIDMap2` lets us attach stable int64 document IDs. We keep the text and payload in a separate metadata table and join returned IDs back to it.

Flat search requires no training; its `train` phase is unnecessary. Check index size and compare the scores against a direct NumPy calculation. Exactness means exact neighbors for these vectors and this metric, not guaranteed relevance to a human question.

In [ ]:
index = faiss.IndexIDMap2(faiss.IndexFlatIP(vectors.shape[1]))
ids = np.ascontiguousarray(catalog.id.to_numpy(),dtype=np.int64)
index.add_with_ids(vectors,ids)
assert index.ntotal == len(catalog)
q = encode_queries([queries.iloc[0]['query']])
scores, found_ids = index.search(q,3)
expected_positions = np.argsort(-(q @ vectors.T)[0])[:3]
assert np.array_equal(found_ids[0],ids[expected_positions])
assert np.allclose(scores[0],(q @ vectors.T)[0,expected_positions],atol=1e-6)
print('Exact search matches the direct dot-product calculation.')

## 6. Turn nearest IDs into a useful search response

A response needs text, IDs and scores, not just an array of neighbors. Limit `k` to the number of indexed points and defensively ignore negative sentinel IDs. A score is a relative similarity under this representation, not a calibrated probability that the document answers the question.

Read the top result and one lower-ranked result. Does compression blur topics? Would a synonym with no shared words still work? This is where representation quality becomes more important than changing the search engine.

In [ ]:
def search(text,k=3):
    if k < 1:
        raise ValueError('k must be positive')
    score, doc_ids = index.search(encode_queries([text]), min(k,index.ntotal))
    rows = []
    for value,doc_id in zip(score[0],doc_ids[0]):
        if doc_id < 0:
            continue
        item = metadata.loc[int(doc_id)]
        rows.append({'id':int(doc_id),'score':float(value),'title':item.title,'text':item.text,'category':item.category})
    return pd.DataFrame(rows)
display(search('duplicate card charge payment twice'))

## 7. Evaluate retrieval separately from vector-search correctness

With one designated relevant article per query, Hit@3 indicates whether it appears in the first three results. Reciprocal rank is `1/rank` when found, otherwise zero; we report MRR@3. These judgments are authored examples, not exhaustive relevance labels.

Compare compressed retrieval with direct TF-IDF cosine retrieval. That comparison asks whether the representation changed relevance. It is different from approximate-index recall, which asks whether an index reproduced an exact vector neighbor set.

The same fixed representation is used for development and test queries; do not keep adjusting SVD dimensions after inspecting the test results.

In [ ]:
rows = []
for row in queries.itertuples(index=False):
    dense_ids = search(row.query,3).id.tolist()
    lexical_scores = (vectorizer.transform([row.query]) @ text_matrix.T).toarray()[0]
    lexical_ids = ids[np.argsort(-lexical_scores)[:3]].tolist()
    for method,result_ids in [('SVD + exact FAISS',dense_ids),('TF-IDF exact',lexical_ids)]:
        rank = result_ids.index(row.relevant_id)+1 if row.relevant_id in result_ids else None
        rows.append({'query':row.query,'split':row.split,'method':method,
                     'hit_at_3':int(rank is not None),'reciprocal_rank_at_3':0 if rank is None else 1/rank})
evaluation = pd.DataFrame(rows)
display(evaluation)
display(evaluation.groupby(['split','method'])[['hit_at_3','reciprocal_rank_at_3']].mean())

## 8. Treat metadata filtering as part of correctness

Our main index was built only from eligible alpha documents. If the user now wants only billing articles, searching the global top three and then removing non-billing results can miss relevant billing articles ranked lower globally.

For this tiny dataset, build an exact index over the eligible category and search it directly. At scale, maintain suitable partitions or use an engine that supports the required filtering. FAISS has index-specific selection capabilities, but it is not a general payload database or authorization layer.

In [ ]:
mask = catalog.category.eq('billing').to_numpy()
billing_index = faiss.IndexIDMap2(faiss.IndexFlatIP(vectors.shape[1]))
billing_index.add_with_ids(vectors[mask],ids[mask])
filtered_scores, filtered_ids = billing_index.search(encode_queries(['refund money card']),min(3,int(mask.sum())))
display(metadata.loc[filtered_ids[0],['title','category']].assign(score=filtered_scores[0]))
assert all(metadata.loc[filtered_ids[0],'category'] == 'billing')

## 9. Measure approximation against an exact neighbor set

Fifteen documents cannot support a meaningful speed claim. To demonstrate index behavior separately, generate 3,000 random 32-dimensional unit vectors and 120 independent queries. These are numerical data, not documents or language embeddings.

Use an exact flat index as the reference. IVF partitions the space into coarse lists, then searches only `nprobe` lists. Training learns the coarse partition; `add` stores the vectors. The training sample should represent future indexed vectors. Higher `nprobe` usually trades more work for higher neighbor recall.

Recall@10 below is overlap with the exact vector top ten. It is not document relevance recall and not a production benchmark.

In [ ]:
rng = np.random.default_rng(42)
base = np.ascontiguousarray(normalize(rng.normal(size=(3000,32))),dtype=np.float32)
benchmark_queries = np.ascontiguousarray(normalize(rng.normal(size=(120,32))),dtype=np.float32)
exact = faiss.IndexFlatIP(32); exact.add(base)
_,truth = exact.search(benchmark_queries,10)
ivf = faiss.IndexIVFFlat(faiss.IndexFlatIP(32),32,32,faiss.METRIC_INNER_PRODUCT)
ivf.train(base); assert ivf.is_trained
ivf.add(base)

def timed_search(engine):
    engine.search(benchmark_queries,10) # warm up before timing
    durations = []
    for _ in range(3):
        start = time.perf_counter(); _,neighbors = engine.search(benchmark_queries,10)
        durations.append(time.perf_counter()-start)
    recall = np.mean([len(set(a) & set(b))/10 for a,b in zip(truth,neighbors)])
    return float(recall), float(np.median(durations)*1000/len(benchmark_queries))

benchmark = []
recall,elapsed = timed_search(exact)
benchmark.append({'index':'FlatIP','nprobe':None,'recall_at_10':recall,'ms_per_query_batch_amortized':elapsed})
for probes in [1,4,16,32]:
    ivf.nprobe = probes
    recall,elapsed = timed_search(ivf)
    benchmark.append({'index':'IVFFlat','nprobe':probes,'recall_at_10':recall,'ms_per_query_batch_amortized':elapsed})
benchmark = pd.DataFrame(benchmark)
display(benchmark)
assert np.isclose(benchmark.iloc[-1].recall_at_10,1.0)

## 10. Compare a graph index without claiming universal speed

HNSW links vectors in a graph. `M` controls graph connectivity, `efConstruction` affects build work, and `efSearch` controls search exploration. This example uses the same vectors and queries as IVF.

Reported milliseconds are batch-amortized, single-process measurements on the current machine, not p95 request latency. Construction time, memory, cold starts and updates are excluded. For a real choice, measure those too on representative data and query/filter distributions.

In [ ]:
hnsw = faiss.IndexHNSWFlat(32,16,faiss.METRIC_INNER_PRODUCT)
hnsw.hnsw.efConstruction = 80
hnsw.add(base)
graph_results = []
for effort in [16,64]:
    hnsw.hnsw.efSearch = effort
    recall,elapsed = timed_search(hnsw)
    graph_results.append({'efSearch':effort,'recall_at_10':recall,'ms_per_query_batch_amortized':elapsed})
display(pd.DataFrame(graph_results))
assert all(0 <= r['recall_at_10'] <= 1 for r in graph_results)
ax = benchmark[benchmark['index']=='IVFFlat'].plot(x='nprobe',y='recall_at_10',marker='o',legend=False)
ax.set(ylabel='Exact-neighbor recall@10',title='Approximation on seeded numerical data',ylim=(0,1.05))
plt.tight_layout(); plt.savefig(OUT / 'ivf_recall.png'); plt.show()

## 11. Save the representation, index and metadata as one version

A FAISS index cannot reconstruct the original article text or the TF-IDF/SVD transform. Save all three with a manifest. On reload, validate count, dimension, IDs and fixed-query results. Only load trusted index and Python serialization files.

Changing the embedding model, normalization or dimensionality requires an appropriate rebuild and re-evaluation. Reusing an old index with a new query encoder can return plausible but meaningless results.

In [ ]:
faiss.write_index(index,str(OUT / 'articles.faiss'))
joblib.dump({'vectorizer':vectorizer,'svd':svd},OUT / 'encoder.joblib')
catalog.to_json(OUT / 'metadata.json',orient='records',indent=2)
manifest = {'representation':'TF-IDF word/bigram + SVD + L2 normalization', 'dimension':int(index.d),
            'metric':'inner product on unit vectors', 'count':int(index.ntotal),'faiss':faiss.__version__,
            'ids':ids.tolist(), 'population':'alpha, published', 'version':'teaching-v1'}
(OUT / 'manifest.json').write_text(json.dumps(manifest,indent=2))
restored = faiss.read_index(str(OUT / 'articles.faiss'))
restored_encoder = joblib.load(OUT / 'encoder.joblib')
restored_metadata = pd.read_json(OUT / 'metadata.json')
restored_query = np.ascontiguousarray(normalize(restored_encoder['svd'].transform(
    restored_encoder['vectorizer'].transform([queries.iloc[0]['query']]))),dtype=np.float32)
a,b_ids = restored.search(restored_query,3)
c,d_ids = index.search(q,3)
assert restored.d == index.d and restored.ntotal == len(restored_metadata)
assert set(restored_metadata.id) == set(ids)
assert np.array_equal(b_ids,d_ids) and np.allclose(a,c)
evaluation.to_csv(OUT / 'relevance_evaluation.csv',index=False)
benchmark.to_csv(OUT / 'approximation_benchmark.csv',index=False)
print('Index, encoder and metadata reload checks passed.')

## 12. Choose an index based on measured constraints

| Index family | Useful property | Tradeoff to examine |
|---|---|---|
| Flat L2 / IP | Exact reference, no training | Scan cost grows with corpus size |
| IVF Flat | Searches selected coarse partitions | Training and `nprobe`/recall tradeoff |
| HNSW | Graph-based approximate search | Graph memory, build work and search effort |
| Product quantization | Compressed vector representations | Quantization error and training requirements |
| ID-mapped wrapper | Stable external IDs | Metadata still lives separately |

Float32 vectors alone require roughly `count × dimension × 4` bytes, before index overhead and metadata. More dimensions can increase both representation capacity and cost; they are not automatically more relevant. GPU support requires a compatible build and hardware and is not part of this CPU tutorial.

A retrieval-augmented generation system would pass retrieved, authorized text into another model. FAISS retrieves; it does not verify answers, provide citations by itself or prevent hallucinations. Evaluate retrieval before adding generation, and keep source IDs through the full pipeline.

## 13. Explain what the evidence supports

1. Design ten additional queries using paraphrases and multiple relevant articles. Keep them untouched while choosing representation settings on development queries.
2. Compare SVD dimensions on development relevance, then report once on a fresh test set. Do not confuse higher exact-neighbor recall with better human relevance.
3. Demonstrate a category query where post-filtering global top-k loses an eligible result. Compare with searching the complete eligible subset.
4. Benchmark a larger representative corpus with build time, memory and tail latency, rather than extrapolating this toy measurement.

| Observation | Investigate |
|---|---|
| High cosine score but wrong article | Encoder quality, corpus wording and relevance labels |
| Different results after reload | Encoder/version mismatch or missing metadata |
| Huge-norm vectors dominate | Missing normalization when cosine was intended |
| Negative returned IDs | Requested k exceeds available neighbors or an IVF probe finds too few |
| Approximate search misses exact neighbors | Search effort, partition quality and index choice |

Official references: [getting started](https://github.com/facebookresearch/faiss/wiki/Getting-started), [metrics](https://github.com/facebookresearch/faiss/wiki/MetricType-and-distances), [IVF search](https://github.com/facebookresearch/faiss/wiki/Faster-search), [index choices](https://github.com/facebookresearch/faiss/wiki/Faiss-indexes), and [index I/O](https://github.com/facebookresearch/faiss/wiki/Index-IO%2C-cloning-and-hyper-parameter-tuning).

## Second project: Keep a filtered catalogue complete during updates

A fictional 3,000-item vector catalogue has malformed vectors, duplicate deliveries and a small eligible tenant. Learn why filtering the global nearest neighbors can silently return too few results, then replace an item without leaving a stale vector.

Run these cells in order. This extension uses its own `p2_` variables and writes only its named outputs under `outputs/project2/`. Predict each comparison before running it; the worked discussion after the code explains the result and its limits.

### Work through eligibility before similarity

Suppose only 1% of items belong to the requested tenant. A global top-10 shortlist contains roughly 0.1 eligible items under independence, even though 30 valid candidates exist. Filtering that shortlist is therefore a recall decision, not merely a presentation step. We compare it with a larger 500-candidate shortlist and exact search over eligible vectors. The exact eligible ranking supplies an independent reference; timing alone cannot establish correctness.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import time
from IPython.display import display
p2_out=Path('outputs/project2');p2_out.mkdir(parents=True,exist_ok=True)

import faiss
faiss.omp_set_num_threads(2)
p2_rng=np.random.default_rng(103)
p2_vectors=p2_rng.normal(size=(3000,32)).astype('float32')
p2_vectors/=np.linalg.norm(p2_vectors,axis=1,keepdims=True)
p2_rows=[(i,v.copy()) for i,v in enumerate(p2_vectors)]
p2_rows += [p2_rows[0],(4001,np.zeros(31,dtype='float32')),(4002,np.full(32,np.nan,dtype='float32'))]
p2_clean={};p2_rejected=[];p2_duplicates=0
for key,vec in p2_rows:
    if vec.shape!=(32,) or not np.isfinite(vec).all() or np.linalg.norm(vec)==0:
        p2_rejected.append(key);continue
    if key in p2_clean:
        if not np.array_equal(p2_clean[key],vec):raise ValueError('Conflicting vector for an existing ID')
        p2_duplicates+=1;continue
    p2_clean[key]=vec
p2_ids=np.array(list(p2_clean),dtype='int64');p2_matrix=np.ascontiguousarray(list(p2_clean.values()),dtype='float32')
p2_index=faiss.IndexIDMap2(faiss.IndexFlatIP(32));p2_index.add_with_ids(p2_matrix,p2_ids)
p2_eligible=p2_ids%100==0
p2_subset=faiss.IndexIDMap2(faiss.IndexFlatIP(32));p2_subset.add_with_ids(p2_matrix[p2_eligible],p2_ids[p2_eligible])
p2_queries=p2_rng.normal(size=(40,32)).astype('float32');p2_queries/=np.linalg.norm(p2_queries,axis=1,keepdims=True)
_,p2_truth=p2_subset.search(p2_queries,5)
assert p2_index.ntotal==3000 and len(p2_rejected)==2 and p2_duplicates==1
print({'accepted':p2_index.ntotal,'eligible':int(p2_eligible.sum()),'rejected_ids':p2_rejected,'duplicate_deliveries':p2_duplicates})

### Compare a bounded search and repair a stale update

For each query, recall@5 is the number of returned IDs in the exact eligible top five divided by five. Returning no candidates scores zero, even if the implementation never raises an exception. Overfetching 500 costs more candidate processing and still has no completeness guarantee. A separate eligible index has maintenance overhead but makes the population explicit.

`add_with_ids` is not an application-level upsert. Remove the old ID before inserting its replacement; verify both population size and the stored vector. In a live service, these operations need a publication or synchronization strategy because readers could observe the gap.

In [ ]:
p2_report=[]
for p2_budget in [10,500]:
    p2_start=time.perf_counter();_,p2_found=p2_index.search(p2_queries,p2_budget)
    p2_filtered=[[int(i) for i in row if i%100==0][:5] for row in p2_found]
    p2_report.append({'approach':f'global top {p2_budget}, then filter','recall_at_5':np.mean([len(set(r)&set(t))/5 for r,t in zip(p2_filtered,p2_truth)]),'mean_returned':np.mean(list(map(len,p2_filtered))),'seconds':time.perf_counter()-p2_start})
p2_start=time.perf_counter();_,p2_exact=p2_subset.search(p2_queries,5)
p2_report.append({'approach':'exact eligible index','recall_at_5':1.,'mean_returned':5.,'seconds':time.perf_counter()-p2_start})
p2_replacement=-p2_matrix[1:2].copy()
assert p2_index.remove_ids(np.array([1],dtype='int64'))==1
p2_index.add_with_ids(p2_replacement,np.array([1],dtype='int64'))
assert p2_index.ntotal==3000
np.testing.assert_allclose(p2_index.reconstruct(1),p2_replacement[0])
faiss.write_index(p2_index,str(p2_out/'updated_catalogue.faiss'))
display(pd.DataFrame(p2_report))

### Decide what the measurement supports

The eligible index examines only 30 vectors, so its advantage here comes partly from selectivity. Thousands of changing tenant indexes can be costly; compare a filter-aware database when that maintenance dominates. These are CPU microbenchmarks with synthetic unit vectors, not embedding relevance measurements.

**Try:** make half the catalogue eligible. **Worked reasoning:** overfetching should lose fewer candidates, while the exact subset grows. Recompute recall and timing together; do not infer a universal threshold from the original 1% case. If an updated item changes tenant, both old and new subset indexes must be updated consistently.